In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'text.usetex': True,
    'font.size': 18
})
print('Code to reproduce the results from experiments in Secion 3.4')
print('Example with skimages Radon transform')
print('')

In [ ]:
# Example from the introduction
# Simple test of the adjoint property of the Radon transform
import numpy as np
from skimage.transform import radon, iradon

np.random.seed(4242) # fix seed for reproducibility, if changed, all results change, especially the ratio
N = 50
numAng = 70
theta = np.linspace(0.0, 180, numAng, endpoint=False)

v = np.random.randn(N,N)
# Set to zero outside reconstruction circle:
mask = np.zeros((N,N), dtype=bool)
X,Y = np.meshgrid(np.arange(N), np.arange(N))
mask[(X-N/2)**2 + (Y-N/2)**2 <= (N/2)**2] = True
v[~mask] = 0
w = np.random.randn(N,numAng)

Av = radon(v, theta=theta, preserve_range=True)
ATw = iradon(w, theta=theta, filter_name=None, preserve_range=True)


print(f'<v,A^Tw> = {np.sum(v*ATw)},  <Av,w> = {np.sum(Av*w)}')
print(f'Quotient: {np.sum(v*ATw)/np.sum(Av*w)}')

In [ ]:
# Further example from the introduction:
# Do power iteration to estimate norm, but use different quantities to estimate it

# define maps on vectorized images
m = N * numAng
n =N**2
def A(v):
    return radon(v.reshape(N,N),theta=theta,preserve_range=True).ravel()
def AT(w):
    # return a 1D array to avoid broadcasting issues
    return iradon(w.reshape(N,numAng),theta=theta,filter_name=None,preserve_range=True).ravel()

print('Return different estimates of the norm of A via power iteration')
v = np.random.randn(n)
v[~mask.ravel()] = 0
v /= np.linalg.norm(v)
for k in range(10): # 10 iterations are enough for stabilization
    Av = A(v)
    ATAv = AT(Av)
    print(f'Step {k}. Est (1): {np.sqrt(np.sum(v*ATAv)):2.4f} Est (2): {np.linalg.norm(ATAv)/np.linalg.norm(Av):2.4f}  Est (3): {np.sqrt(np.linalg.norm(ATAv)):2.4f} Est (4): {np.linalg.norm(Av):2.4f}') 
    v = ATAv/np.linalg.norm(ATAv)

vPower = v
uPower = A(vPower)
uPower /= np.linalg.norm(uPower)
normAPower = np.linalg.norm(A(vPower))
normATPower = np.linalg.norm(AT(uPower))

In [ ]:
# Example from Section 3.4

# different geometry (all geometries lead to basically comparable results)
N = 125
numAng = 6
theta = np.linspace(0.0, 180, numAng, endpoint=False)
def A(v):
    return radon(v.reshape(N,N),theta=theta,preserve_range=True).ravel()
def AT(w):
    # return a 1D array to avoid broadcasting issues
    return iradon(w.reshape(N,numAng),theta=theta,filter_name=None,preserve_range=True).ravel()

# Recompute mask
mask = np.zeros((N,N), dtype=bool)
X,Y = np.meshgrid(np.arange(N), np.arange(N))
mask[(X-N/2)**2 + (Y-N/2)**2 <= (N/2)**2] = True

# Do power iteration to estimate norm, but use different quantities to estimate it
m = N * numAng
n =N**2
print('Return different estimates of the norm of A via power iteration')
v = np.random.randn(n)
v[~mask.ravel()] = 0
v /= np.linalg.norm(v)
for k in range(10): # 10 iterations are enough for stabilization
    Av = A(v)
    ATAv = AT(Av)
    print(f'Step {k}. Est (1): {np.sqrt(np.sum(v*ATAv)):2.4f} Est (2): {np.linalg.norm(ATAv)/np.linalg.norm(Av):2.4f}  Est (3): {np.sqrt(np.linalg.norm(ATAv)):2.4f} Est (4): {np.linalg.norm(Av):2.4f}') 
    v = ATAv/np.linalg.norm(ATAv)

vPower = v
uPower = A(vPower)
uPower /= np.linalg.norm(uPower)
normAPower = np.linalg.norm(A(vPower))
normATPower = np.linalg.norm(AT(uPower))

In [ ]:
# Apply our method to estimate norm of A, use the approximate right singular vector from
# power method as initialization
print('Estimate norm via our method')
v = vPower.copy()
Av = A(v)
normAv = np.linalg.norm(Av)

n = np.prod(v.shape)
m = np.prod(Av.shape)

print(f'Radon transform, input dim {n}, output dim {m}')

# list of estimates
nA = []

# list of stepsizes
taus = []

nA.append(normAv)
maxiter = n
check = 100
print(f'Run for {maxiter} iterations')

for i in range(maxiter):
    # sample x orthogonal to v with unit norm
    x = np.random.randn(n)
    x[~mask.ravel()] = 0
    x -= np.sum(x*v)*v
    x /= np.linalg.norm(x)
    Ax = A(x)
    normAx = np.linalg.norm(Ax)
    vOld = v.copy()
    a = np.sum(Ax*Av)
    b = normAx**2 - normAv**2
    tau = np.sign(a)*(b/(2*np.abs(a)) + np.sqrt(b**2/(4*a**2)+1))
    v += tau*x
    v /= np.linalg.norm(v)
    Av = A(v)
    normAv = np.linalg.norm(Av)
    nA.append(normAv)
    taus.append(tau)
    #aS.append(a)
    if i%check==0:
        print(f'Step {i: 5d}. Est: {nA[-1]:2.4f}, (step size {tau:>7.2e}, a^2 {a**2:>7.2e})')

vAdjointFree = v
normA = nA[-1]



In [ ]:
#
# Do some plots of right singular vectors:
# produce figure 7 of the paper (without colorbars)
fig, ax = plt.subplots(1,3, figsize=(8,6))
ax[0].imshow(v.reshape(N,N)*mask)
ax[1].imshow(vPower.reshape(N,N)*mask)
ax[2].imshow(np.abs(v-vPower).reshape(N,N)*mask)
plt.show()

In [ ]:
# Apply our method to estimate norm of AT, use the approximate right singular vector from
# power method as initialization
print('Estimate norm via our method')
u = uPower.copy()
ATu = AT(u)
normATu = np.linalg.norm(ATu)

n_ = np.prod(u.shape)
m_ = np.prod(ATu.shape)
print(f'adjoint Radon transform, input dim {n_}, output dim {m_}')

# list of estimates
nAT = []

# list of stepsizes
taus = []

nAT.append(normATu)
maxiter = 500 # only do 500 iterations since nothing is changing
check = 100
print(f'Run for {maxiter} iterations')

for i in range(maxiter):
    # sample x orthogonal to v with unit norm
    x = np.random.randn(n_)
    x -= np.sum(x*u)*u
    x /= np.linalg.norm(x)
    ATx = AT(x)
    normATx = np.linalg.norm(ATx)
    uOld = u.copy()
    a = np.sum(ATx*ATu)
    b = normATx**2 - normATu**2
    tau = np.sign(a)*(b/(2*np.abs(a)) + np.sqrt(b**2/(4*a**2)+1))
    u += tau*x
    u /= np.linalg.norm(u)
    ATu = AT(u)
    normATu = np.linalg.norm(ATu)
    nAT.append(normATu)
    taus.append(tau)
    #aS.append(a)
    if i%check==0:
        print(f'Step {i: 5d}. Est: {nAT[-1]:2.4f}, (step size {tau:2.2e}, ak^2 = {a**2:2.2e})')

uAdjointFree = u    
normAT = nAT[-1]